# M4 — Median-beat / template detector (combined PTB-XL + Ningbo)

Per lead: detect R (delineation-free, **multi-lead composite**), **cross-correlation-align** beats, build a **median beat** (persistent WPW) + a **most-pre-excited beat** (intermittent) + an **inter-beat variability** view (MAD across beats + per-zone MAD + delta-fraction), extract **calibration-invariant morphology** normalized by **|R| amplitude** (delta/onset multi-zone, PR, QRS width, ST/T, shape samples; delineation-free). RR/HR excluded (batch confound).

**ONE Run All = A/B of BOTH R-detectors (`pan_tompkins` + `wavelet_env`) + rigor controls.** XGBoost only, NaN-native, Platt. **Judge = pooled OOF AP (folds 1-8); fold 9 = validation; fold 10 UNTOUCHED.** Figures = canonical AP-vs-k. Deployed artifact = `m4_combined_*`.

> **Frozen-artifact mode.** Block 8a.2 reloads the frozen model and out-of-fold scores when they exist, instead of refitting. This avoids run-to-run XGBoost drift (`n_jobs`/threading) that perturbs the marginal fold-9 confusion matrix. Frozen values: OOF AP 0.718, K=220, TP 10 / FN 3 (`wavelet_env`); 0.705 (`pan_tompkins`). The fit runs only if the frozen artifacts are absent. Every other step (gate, ablation, AP-vs-k, grid) reloads its cache in the same way.

> **DATA REQUIREMENT.** Run All needs the full feature matrices in `data/processed/` (`m4_features_pan_tompkins.csv`, `m4_features_wavelet_env.csv`; several GB each). They are not distributed with this repository and are regenerated by the extraction blocks of this notebook. Block 8a.2 raises an explicit error if a matrix is incomplete.


### Structure & rigor

8a.0-8a.1 helpers (R-detect both, cross-corr align, |R|-norm morphology: med/ext/var) + synthetic test; 8a.1c PTB demographics for the confound; **8a.2 `run_detector(DET)`** (chunked disk-write extraction → gate FDR+cross-dataset → family ablation → numpy AP-vs-k+CI → depth-open K grid + parsimony tiebreak → **reload frozen v1 model+oof (or refit if absent)** → eval → per-dataset + dataset + age/sex confound → ensemble preview vs M1/M2/M3 → freeze); **8a.3 driver (both detectors)**; **8a.4 permutation negative-control**; **8a.5 R-detector concordance**; **8a.6 verdict + promote → `m4_combined_*`**. Caches resumable; fold 10 never touched.

### Block 8a.0: Setup — detectors (A/B), frozen filter, beat window, feature config, imports

In [ ]:
# M4 - MEDIAN-BEAT detector, COMBINED (v1 representation). ONE Run All = A/B of the two R-detectors
# (pan_tompkins + wavelet_env) + rigor controls. Representation: 3 beat-views per lead -> MEDIAN (persistent
# WPW) + MOST-PRE-EXCITED (intermittent) + inter-beat VARIABILITY; calibration-invariant morphology normalized
# by |R| amplitude; delineation-free (R-peak only). RR/HR excluded (batch confound). Judge: AP OOF folds 1-8;
# fold9 = validation; fold 10 UNTOUCHED.
# NOTE: the frozen numbers (OOF 0.718) are reloaded from the cached artifacts by the guards on Run All; the
# extraction code below only executes if the feature cache is absent.
import os, sys, json, warnings, time, contextlib, glob
import numpy as np, pandas as pd, pywt
from scipy.signal import butter, sosfiltfilt, find_peaks
from scipy.stats import skew, mannwhitneyu
from statsmodels.stats.multitest import multipletests
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from joblib import Parallel, delayed
from tqdm import tqdm
import matplotlib.pyplot as plt
import joblib
from datetime import datetime
warnings.filterwarnings('ignore'); EPS=1e-9
DETECTORS=['pan_tompkins','wavelet_env']   # A/B both R-detectors; 8a.6 promotes the winner to canonical m4_combined_*
ROOT=r'.'
PROCESSED=os.path.join(ROOT,'data','processed'); MODELS=os.path.join(ROOT,'models','M4_medianbeat')
FIG=os.path.join(ROOT,'reports','figures'); METRICS=os.path.join(ROOT,'reports','metrics'); SRC=os.path.join(ROOT,'src')
RAW=os.path.join(ROOT,'data','raw')
for d in (MODELS,FIG,METRICS): os.makedirs(d,exist_ok=True)
sys.path.insert(0,SRC)
from signal_loading import load_signal, LEADS_CANONICAL
from evaluation import evaluate_standard
with open(os.path.join(PROCESSED,'filter_config.json'),encoding='utf-8') as f: FCFG=json.load(f)['filter_FINAL']
assert (FCFG['low'],FCFG['high'],FCFG['order'])==(0.5,40,4), "Unexpected filter!"
FS=FCFG['fs']; LEADS_M4=list(LEADS_CANONICAL); LEAD_IDX={L:LEADS_CANONICAL.index(L) for L in LEADS_M4}
TAG='combined'; TIE_EPS=0.01
RPRE,RPOST=int(0.25*FS),int(0.45*FS); SHAPE_N,ACCEL_N=50,30; MIN_BEATS=3; PAD=5120   # window [-250,+450] ms
SEG={'P':(-250,-120),'PR':(-120,-40),'onQRS':(-40,0),'bodyQRS':(0,40),'offQRS':(40,80),'ST':(80,250),'T':(250,450)}
ONSET_REG=[(-100,-40),(-80,-20),(-60,-10)]
SOS_BP=butter(FCFG['order'],[FCFG['low']/(FS/2),FCFG['high']/(FS/2)],btype='band',output='sos')
def bp(x): return sosfiltfilt(SOS_BP,np.asarray(x,dtype=np.float64))
meta=pd.read_csv(os.path.join(PROCESSED,'metadata_combined.csv'),dtype={'ecg_id':str})
assert int((meta.label==1).sum())==142, "Expected 142 WPW!"
print(f"M4 v1 | leads {len(LEADS_M4)} | filter {FCFG['low']}-{FCFG['high']}Hz | detectors {DETECTORS}")
print(f"metadata: {len(meta)} ECGs, 142 WPW | Train(1-8) {int(meta[(meta.fold.between(1,8))&(meta.label==1)].shape[0])} / Val(9) {int(meta[(meta.fold==9)&(meta.label==1)].shape[0])} / Test(10) {int(meta[(meta.fold==10)&(meta.label==1)].shape[0])} [UNTOUCHED]")
print("Block 8a.0 OK.")

### Block 8a.0b: Patient-leakage check across folds (blocking assert)

In [ ]:
# Patient-leakage check across folds (blocking assert).
pat=meta.groupby('patient_id')['fold'].nunique(); leaky=pat[pat>1]
print(f"Unique patients: {meta.patient_id.nunique()} | ECGs: {len(meta)} | in >1 fold: {len(leaky)}")
assert len(leaky)==0, f"PATIENT LEAK: {len(leaky)} patients in multiple folds."
print("[OK] No patient leakage across folds.")

### Block 8a.1: Helpers (R-detect, cross-corr align, |R|-norm morphology: shape/seg/onset + variability view) + synthetic test

In [ ]:
# Helpers (detector-independent). R-detect (composite multi-lead RMS -> pan_tompkins OR wavelet_env), cross-corr
# align, |R|-amplitude normalization (v1, calibration-invariant), morphology on 3 beat-views: MEDIAN,
# MOST-PRE-EXCITED (max onset slur), and inter-beat VARIABILITY (MAD across beats + per-zone MAD + deltafrac).
# No wavelet of the beat (=M3 boundary), no inter-lead (=M5 boundary), RR/HR excluded. n_beats = ECG-level QC only.
@contextlib.contextmanager
def tqdm_joblib(t):
    class _Cb(joblib.parallel.BatchCompletionCallBack):
        def __call__(self,*a,**k): t.update(n=self.batch_size); return super().__call__(*a,**k)
    old=joblib.parallel.BatchCompletionCallBack; joblib.parallel.BatchCompletionCallBack=_Cb
    try: yield t
    finally: joblib.parallel.BatchCompletionCallBack=old; t.close()
def _pad(x): n=len(x); return x[:PAD] if n>=PAD else np.pad(x,(0,PAD-n),mode='reflect')
def _pt_rpeaks(comp):
    sos=butter(2,[5/(FS/2),15/(FS/2)],btype='band',output='sos'); fsig=sosfiltfilt(sos,comp)
    sq=np.gradient(fsig)**2; w=max(1,int(0.15*FS)); mwi=np.convolve(sq,np.ones(w)/w,mode='same')
    pk,_=find_peaks(mwi,distance=int(0.25*FS),height=np.mean(mwi)+0.5*np.std(mwi)); return pk
def _we_rpeaks(comp):
    cf=pywt.swt(_pad(comp),'db4',level=6,trim_approx=True,norm=True)
    D3,D4=np.asarray(cf[4])[:len(comp)],np.asarray(cf[3])[:len(comp)]
    w=max(1,int(0.04*FS)); env=np.convolve(np.abs(D3)+np.abs(D4),np.ones(w)/w,mode='same')
    pk,_=find_peaks(env,distance=int(0.25*FS),height=np.mean(env)+0.5*np.std(env)); return pk
def detect_r(bp12, DET):
    comp=np.sqrt((bp12**2).mean(axis=1)); pk=_pt_rpeaks(comp) if DET=='pan_tompkins' else _we_rpeaks(comp)
    w=int(0.05*FS); out=[]
    for p in pk:
        a,b=max(0,p-w),min(len(comp),p+w)
        if b>a: out.append(int(a+np.argmax(comp[a:b])))
    return np.array(sorted(set(out)))
def _sl(m0,m1): return slice(RPRE+int(m0/1000*FS), RPRE+int(m1/1000*FS))
def _align(b,ref,maxlag=10):
    c=np.correlate(b-b.mean(),ref-ref.mean(),'full'); mid=len(b)-1
    lag=int(np.argmax(c[mid-maxlag:mid+maxlag+1]))-maxlag
    return np.roll(b,-lag)
def _seg_stats(b,p,o):
    for nm,(m0,m1) in SEG.items():
        s=b[_sl(m0,m1)]
        if len(s)<2: continue
        o[f'{p}_{nm}_area']=float(np.trapz(s)); o[f'{p}_{nm}_slope']=float(np.polyfit(np.arange(len(s)),s,1)[0])
        o[f'{p}_{nm}_max']=float(s.max()); o[f'{p}_{nm}_min']=float(s.min()); o[f'{p}_{nm}_rms']=float(np.sqrt(np.mean(s**2)))
        o[f'{p}_{nm}_skew']=float(skew(s)) if len(s)>3 else np.nan
def beat_features(b,p):                                                            # v1: shape + seg + onset (NO cum/notch/delta-angle)
    o={}; v=np.gradient(b); ac=np.gradient(v)
    for i,ix in enumerate(np.linspace(0,len(b)-1,SHAPE_N).astype(int)): o[f'{p}_amp{i}']=float(b[ix])
    for i,ix in enumerate(np.linspace(0,len(v)-1,SHAPE_N).astype(int)): o[f'{p}_vel{i}']=float(v[ix])
    for i,ix in enumerate(np.linspace(0,len(ac)-1,ACCEL_N).astype(int)): o[f'{p}_acc{i}']=float(ac[ix])
    _seg_stats(b,p,o); vmax=np.max(np.abs(v))+EPS
    for j,(m0,m1) in enumerate(ONSET_REG):
        s=b[_sl(m0,m1)]
        if len(s)<2: continue
        o[f'{p}_on{j}_slope']=float(np.polyfit(np.arange(len(s)),s,1)[0]); o[f'{p}_on{j}_area']=float(np.trapz(np.abs(s)))
        o[f'{p}_on{j}_signed']=float(np.mean(s)); o[f'{p}_on{j}_slur']=float(np.mean(np.abs(np.gradient(s)))/vmax)
    P=b[_sl(-250,-120)]
    o[f'{p}_PR_ms']=float((len(P)-int(np.argmax(P)))*1000/FS+120) if len(P)>2 else np.nan
    thr=0.2*vmax; o[f'{p}_qrswidth_ms']=float(np.sum(np.abs(v[_sl(-60,80)])>thr)*1000/FS); o[f'{p}_pol_onset']=float(np.mean(b[_sl(-40,0)]))
    return o
def _delta_score(b):                                                              # onset slur (intermittency proxy)
    s=b[_sl(-80,-20)]; v=np.gradient(b); vmax=np.max(np.abs(v))+EPS
    return float(np.mean(np.abs(np.gradient(s)))/vmax) if len(s)>=2 else 0.0
def beat_var_features(Bn,p):                                                       # v1 'var' view
    o={}; med=np.median(Bn,axis=0); mad_all=np.median(np.abs(Bn-med),axis=0)
    for i,ix in enumerate(np.linspace(0,Bn.shape[1]-1,SHAPE_N).astype(int)): o[f'{p}_mad{i}']=float(mad_all[ix])   # MAD across beats per sample
    for nm,(m0,m1) in SEG.items():
        sl=_sl(m0,m1)
        if sl.stop-sl.start<2: continue
        ar=np.array([float(np.trapz(bb[sl])) for bb in Bn]); o[f'{p}_{nm}_mad']=float(np.median(np.abs(ar-np.median(ar)))) if len(ar)>1 else 0.0
    sc=np.array([_delta_score(bb) for bb in Bn]); thr=np.median(sc)+0.5*(np.std(sc)+EPS)
    o[f'{p}_deltafrac']=float(np.mean(sc>thr)) if len(sc)>1 else 0.0              # fraction of beats more pre-excited than typical
    return o
def _onset_metric(b): s=b[_sl(-60,-10)]; return float(np.trapz(np.abs(s))) if len(s)>=2 else 0.0
def _normfac(bb): r=abs(float(bb[RPRE])); return r if r>EPS else EPS              # |R| amplitude (v1)
def extract_lead(sig,rpk,L):
    segs=[sig[r-RPRE:r+RPOST] for r in rpk if r-RPRE>=0 and r+RPOST<=len(sig)]
    if len(segs)<MIN_BEATS: return None
    B=np.array(segs); med0=np.median(B,axis=0)
    keep=[i for i,bb in enumerate(B) if np.corrcoef(bb,med0)[0,1]>0.5]
    if len(keep)<MIN_BEATS: keep=list(range(len(B)))
    B=np.array([_align(bb,med0,10) for bb in B[keep]])                             # cross-corr realign
    Bn=np.array([bb/_normfac(bb) for bb in B])                                     # |R| normalization (v1)
    med=np.median(Bn,axis=0); oms=np.array([_onset_metric(bb) for bb in Bn]); ext=Bn[int(np.argmax(oms))]
    o={}; o.update(beat_features(med,f'med_{L}')); o.update(beat_features(ext,f'ext_{L}')); o.update(beat_var_features(Bn,f'var_{L}'))
    return o
def process_one(m, DET):
    warnings.filterwarnings('ignore')
    row={'ecg_id':m['ecg_id'],'patient_id':m['patient_id'],'label':m['label'],'fold':m['fold'],'source':m['source'],'extraction_failed':0,'n_beats':0}
    try:
        raw=load_signal(m['ecg_id'],m['source']); bp12=np.column_stack([bp(raw[:,LEAD_IDX[L]]) for L in LEADS_M4])
        rpk=detect_r(bp12,DET); row['n_beats']=int(len(rpk))
        if len(rpk)<MIN_BEATS: row['extraction_failed']=1; return row
        ok=False
        for L in LEADS_M4:
            f=extract_lead(bp12[:,LEAD_IDX[L]],rpk,L)
            if f is None: continue
            row.update(f); ok=True
        if not ok: row['extraction_failed']=1
    except Exception: row['extraction_failed']=1
    return row
def cohens_d(a,b):
    a,b=a[~np.isnan(a)],b[~np.isnan(b)]
    if len(a)<2 or len(b)<2: return np.nan
    sp=np.sqrt(((len(a)-1)*a.var(ddof=1)+(len(b)-1)*b.var(ddof=1))/(len(a)+len(b)-2))
    return (a.mean()-b.mean())/sp if sp>0 else np.nan
def d_ci(a,b,n=1000,seed=42):
    rng=np.random.default_rng(seed); a,b=a[~np.isnan(a)],b[~np.isnan(b)]
    if len(a)<2 or len(b)<2: return (np.nan,np.nan)
    ds=[cohens_d(rng.choice(a,len(a),True),rng.choice(b,len(b),True)) for _ in range(n)]
    return tuple(np.nanpercentile(ds,[2.5,97.5]))
def ap_boot(yy,pp,n=200,seed=42):
    rng=np.random.default_rng(seed); pos,neg=np.where(yy==1)[0],np.where(yy==0)[0]; a=np.empty(n)
    for i in range(n):
        idx=np.concatenate([rng.choice(pos,len(pos),True),rng.choice(neg,len(neg),True)]); a[i]=average_precision_score(yy[idx],pp[idx])
    return np.median(a),np.percentile(a,2.5),np.percentile(a,97.5)
def make_xgb(spw,**kw):
    p=dict(n_estimators=100,max_depth=2,learning_rate=0.05,subsample=0.8,colsample_bytree=0.8,reg_lambda=2.0,
           min_child_weight=3,scale_pos_weight=spw,eval_metric='aucpr',tree_method='hist',random_state=42,n_jobs=6)
    p.update(kw); return XGBClassifier(**p)
def _view(c): return ('median' if c.startswith('med_') else 'extreme' if c.startswith('ext_') else 'variability' if c.startswith('var_') else '?')
def _family(c):
    if ('_mad' in c) or ('_deltafrac' in c): return 'variability'
    if any(t in c for t in ('_on0','_on1','_on2','_onQRS','_pol_onset')): return 'delta_onset'
    if ('_PR_' in c) or ('_P_' in c): return 'PR'
    if ('_ST_' in c) or ('_T_' in c): return 'ST_T'
    if ('_bodyQRS' in c) or ('_offQRS' in c) or ('_qrswidth' in c): return 'QRS'
    if any(t in c for t in ('_amp','_vel','_acc')): return 'shape'
    return 'other'
print("Block 8a.1 helpers. Synthetic test...")
rng=np.random.default_rng(0); sig=np.zeros((5000,12))
for L in range(12):
    s=0.1*rng.standard_normal(5000)
    for r in range(300,4800,650):
        s[r-180:r-120]+=0.15*np.hanning(60); s[r-40:r]+=np.linspace(0,0.3,40); s[r-3:r+3]+=1.0; s[r+120:r+240]+=0.2*np.hanning(120)
    sig[:,L]=s
bp12=np.column_stack([bp(sig[:,L]) for L in range(12)]); rpk=detect_r(bp12,'pan_tompkins')
fe=extract_lead(bp12[:,1],rpk,'II'); nlead=len(fe) if fe else 0
print(f"  R {len(rpk)} (~8) | features/lead {nlead} (med/ext/var, expect ~432) -> ~{nlead*12}/ECG")
assert fe and all(np.isfinite(v) or (isinstance(v,float) and np.isnan(v)) for v in fe.values()), "bad features!"

### Block 8a.1c: Demographics build (PTB-XL age/sex) for the confound probe

In [ ]:
# Demographics for the age/sex confound (best-effort): PTB-XL via ptbxl_database.csv (our patient_id suffix
# = their patient_id, 100% match) -> {our ecg_id: (age, sex)}. Ningbo age/sex deferred (.hea parse, lower value).
DEMO={}
try:
    db=pd.read_csv(glob.glob(os.path.join(RAW,'ptbxl','ptb-xl-*','ptbxl_database.csv'))[0])
    pa={int(r['patient_id']):(float(r['age']),float(r['sex'])) for _,r in db.iterrows() if pd.notna(r['patient_id'])}
    for _,r in meta[meta.source=='ptbxl'].iterrows():
        try: suf=int(str(r['patient_id']).replace('ptbxl_',''))
        except Exception: continue
        if suf in pa: DEMO[r['ecg_id']]=pa[suf]
    print(f"demographics: PTB-XL age/sex mapped for {len(DEMO)} ECGs (Ningbo deferred). age/sex confound active on PTB subjects.")
except Exception as e:
    print(f"demographics build failed ({e}) -> age/sex confound skipped.")

### Block 8a.2: run_detector(DET) — full v1 pipeline (extraction → gate → ablation → AP-vs-k → K grid → reload frozen model (or refit) → eval → confounds → preview → freeze)

In [ ]:
# Full per-detector pipeline (v1). Caches suffixed by DET -> resumable; the guards reload cached artifacts so
# Run All reproduces the frozen result (OOF 0.718) without re-extracting. Engineering (result-identical, faster):
# each worker writes its chunk to disk in float32; features read as float32; modeling on a numpy matrix; n_jobs=6.
def run_detector(DET):
    print(f"\n{'='*72}\n### DETECTOR: {DET}\n{'='*72}")
    FEATURES_CSV=os.path.join(PROCESSED,f'm4_features_{DET}.csv')
    if os.path.exists(FEATURES_CSV):
        print(f"[guard] {os.path.basename(FEATURES_CSV)} exists -> extraction skipped (cached).")
    else:
        import shutil as _sh
        recs=meta.to_dict('records'); t0=time.time()
        _=Parallel(n_jobs=4,backend='loky')(delayed(process_one)(m,DET) for m in recs[:200])
        print(f"  ETA full ~ {(time.time()-t0)/200*len(recs)/60:.0f} min")
        tmpdir=os.path.join(PROCESSED,f'_m4tmp_{DET}')
        if os.path.exists(tmpdir): _sh.rmtree(tmpdir)
        os.makedirs(tmpdir)
        def _proc_chunk(j,ms,DET,tmpdir):
            M0=['ecg_id','patient_id','label','fold','source','extraction_failed','n_beats']
            rws=[process_one(m,DET) for m in ms]; d=pd.DataFrame(rws)
            fc=[c for c in d.columns if c not in M0]
            if fc: d[fc]=d[fc].astype('float32')
            d.to_csv(os.path.join(tmpdir,f'part_{j:05d}.csv'),index=False); return j
        chunks=[recs[i:i+500] for i in range(0,len(recs),500)]; t0=time.time()
        with tqdm_joblib(tqdm(total=len(chunks),desc=f'M4 extraction ({DET})',unit='chunk')):
            Parallel(n_jobs=6,backend='loky')(delayed(_proc_chunk)(j,c,DET,tmpdir) for j,c in enumerate(chunks))
        parts=sorted(glob.glob(os.path.join(tmpdir,'part_*.csv'))); cols=pd.read_csv(parts[0],nrows=0).columns.tolist(); first=True
        for pp in parts:
            dd=pd.read_csv(pp).reindex(columns=cols); dd.to_csv(FEATURES_CSV,index=False,mode=('w' if first else 'a'),header=first); first=False
        _sh.rmtree(tmpdir); print(f"  extraction done in {(time.time()-t0)/60:.1f} min ({len(parts)} chunks)")
    _hdr=pd.read_csv(FEATURES_CSV,nrows=0).columns.tolist()
    _sc={'ecg_id','patient_id','source'}; _ic={'label','fold','extraction_failed','n_beats'}
    df=pd.read_csv(FEATURES_CSV,dtype={c:(str if c in _sc else 'int16' if c in _ic else 'float32') for c in _hdr})
    METAC=['ecg_id','patient_id','label','fold','source','extraction_failed','n_beats']
    ALL_FEATS=[c for c in df.columns if c not in METAC]; tr=df[df.fold.between(1,8)]; failrate=float(df.extraction_failed.mean())
    print(f"  features {len(ALL_FEATS)} | train1-8 {len(tr)} ({int((tr.label==1).sum())} WPW) | failrate {100*failrate:.1f}%")
    if int((tr.label==1).sum())<100:
        raise ValueError(f"Features CSV for {DET} INCOMPLETE: train1-8={len(tr)} rows / {int((tr.label==1).sum())} WPW (expected ~53540 / 115). Regenerate it with the extraction block above, or restore a complete copy into data/processed/.")
    SEL=os.path.join(PROCESSED,f'm4_{TAG}_selection_{DET}.csv')
    if os.path.exists(SEL) and set(pd.read_csv(SEL,usecols=['feature'])['feature'])==set(ALL_FEATS):
        res=pd.read_csv(SEL); print(f"  [guard] gate reloaded ({len(res)})")
    else:
        w,n=tr[tr.label==1],tr[tr.label==0]; ptb,nin=tr[tr.source=='ptbxl'],tr[tr.source=='ningbo']; rows=[]
        for c in tqdm(ALL_FEATS,desc=f'Gate ({DET})',unit='feat'):
            a,b=w[c].values.astype(float),n[c].values.astype(float); d=cohens_d(a,b)
            try: _,p=mannwhitneyu(a[~np.isnan(a)],b[~np.isnan(b)],alternative='two-sided')
            except Exception: p=np.nan
            lo,hi=d_ci(a,b)
            dp=cohens_d(ptb[ptb.label==1][c].values.astype(float),ptb[ptb.label==0][c].values.astype(float))
            dn=cohens_d(nin[nin.label==1][c].values.astype(float),nin[nin.label==0][c].values.astype(float))
            cross_ok=(np.isfinite(dp) and np.isfinite(dn) and np.sign(dp)==np.sign(dn) and abs(dp)>0.2 and abs(dn)>0.2)
            ci_ok=(np.isfinite(lo) and np.isfinite(hi) and (lo>0)==(hi>0))
            rows.append({'feature':c,'d':d,'d_ptb':dp,'d_nin':dn,'p_raw':p,'ci_excl0':ci_ok,'cross_ok':cross_ok})
        res=pd.DataFrame(rows); ok=res.p_raw.notna(); res.loc[ok,'p_FDR']=multipletests(res.loc[ok,'p_raw'],method='fdr_bh')[1]
        res['gate']=(res.d.abs()>0.3)&(res.p_FDR<0.05)&res.ci_excl0&res.cross_ok
        res=res.sort_values('d',key=lambda s:s.abs(),ascending=False).reset_index(drop=True); res.to_csv(SEL,index=False)
    passed=res[res.gate].feature.tolist(); print(f"  gate-passed {len(passed)}")
    def dedup_fast(feats,thr=0.95):
        sub=tr[feats].rank().to_numpy(); cm=np.nanmean(sub,axis=0); ii=np.where(np.isnan(sub)); sub[ii]=np.take(cm,ii[1])
        C=np.abs(np.corrcoef(sub,rowvar=False)); idx={f:i for i,f in enumerate(feats)}; keep=[]
        for f in feats:
            if all(C[idx[f],idx[k]]<=thr for k in keep): keep.append(f)
        return keep
    dedup_list=dedup_fast(passed) if passed else []
    print(f"  dedup>0.95 -> {len(dedup_list)} | by view {pd.Series(dedup_list).map(_view).value_counts().to_dict()}")
    d8=df[df.fold.between(1,8)].reset_index(drop=True); y=d8.label.values; folds=d8.fold.values; src8=d8.source.values; eid8=d8.ecg_id.values
    FOLD_MASKS=[(folds!=h,folds==h) for h in sorted(np.unique(folds))]
    f9=df[df.fold==9].reset_index(drop=True); y9=f9.label.values; spw8=(y==0).sum()/max((y==1).sum(),1)
    COL={c:i for i,c in enumerate(dedup_list)}; Xall8=d8[dedup_list].to_numpy(dtype=np.float32); Xall9=f9[dedup_list].to_numpy(dtype=np.float32)
    def _ci(feats): return [COL[c] for c in feats]
    def oof_xgb(feats,**kw):
        X=Xall8[:,_ci(feats)]; oof=np.full(len(d8),np.nan)
        for trm,vam in FOLD_MASKS:
            if y[trm].sum()==0 or y[vam].sum()==0: continue
            oof[vam]=make_xgb((y[trm]==0).sum()/max((y[trm]==1).sum(),1),**kw).fit(X[trm],y[trm]).predict_proba(X[vam])[:,1]
        return oof
    def oof_ap(feats,**kw):
        oof=oof_xgb(feats,**kw); m=~np.isnan(oof); return float(average_precision_score(y[m],oof[m]))
    def diag_one(cfg,feats):
        ci=_ci(feats); X8=Xall8[:,ci]; X9=Xall9[:,ci]; oof=np.full(len(d8),np.nan)
        for trm,vam in FOLD_MASKS:
            if y[trm].sum()==0 or y[vam].sum()==0: continue
            oof[vam]=make_xgb((y[trm]==0).sum()/max((y[trm]==1).sum(),1),**cfg).fit(X8[trm],y[trm]).predict_proba(X8[vam])[:,1]
        m=~np.isnan(oof); ao=average_precision_score(y[m],oof[m]); mdl=make_xgb(spw8,**cfg).fit(X8,y)
        at=average_precision_score(y,mdl.predict_proba(X8)[:,1]); af=average_precision_score(y9,mdl.predict_proba(X9)[:,1]) if y9.sum()>0 else np.nan
        return ao,at,at-ao,af
    def oof_perfold_ap(feats,**kw):
        oof=oof_xgb(feats,**kw); aps=[]
        for h in sorted(np.unique(folds)):
            msk=(folds==h)&(~np.isnan(oof))
            if y[msk].sum()>0: aps.append(average_precision_score(y[msk],oof[msk]))
        return np.array(aps)
    ABL=os.path.join(PROCESSED,f'm4_{TAG}_familyablation_{DET}.csv'); FAMS={}
    for c in passed: FAMS.setdefault(_family(c),[]).append(c)
    if os.path.exists(ABL): qc=pd.read_csv(ABL)
    else:
        rowsA=[]
        for name,feats in list(FAMS.items())+[('ALL',passed)]:
            dl=dedup_fast(feats)
            if len(dl)<2: continue
            for k in sorted(set(list(range(10,len(dl)+1,30))+[len(dl)])): rowsA.append({'family':name,'k':k,'AP':oof_ap(dl[:k]),'n':len(dl)})
        qc=pd.DataFrame(rowsA); qc.to_csv(ABL,index=False)
    plt.figure(figsize=(12,6))
    for name in qc['family'].unique():
        cc=qc[qc.family==name]; plt.plot(cc.k,cc.AP,'o-',ms=3,label=f"{name} ({cc.AP.max():.3f})"); print(f"  fam {name:12s}: best OOF AP {cc.AP.max():.3f}")
    for v in (0.198,0.299,0.619): plt.axhline(v,ls='--',color='gray',lw=.7)
    plt.xlabel('k'); plt.ylabel('AP OOF'); plt.legend(fontsize=8); plt.grid(alpha=.3); plt.title(f'M4 family ablation ({DET})')
    plt.savefig(os.path.join(FIG,f'm4_{TAG}_family_ablation_{DET}.png'),dpi=140,bbox_inches='tight'); plt.show()
    RK=os.path.join(PROCESSED,f'm4_{TAG}_ap_vs_k_{DET}.csv'); CIp=os.path.join(PROCESSED,f'm4_{TAG}_ap_vs_k_ci_{DET}.csv')
    if os.path.exists(RK): rk=pd.read_csv(RK)
    else:
        rows=[]; ks=list(range(1,len(dedup_list)+1,2))
        for ii,k in enumerate(tqdm(ks,desc=f'AP-vs-k ({DET})',unit='k')):
            rows.append({'k':int(k),'AP_oof':oof_ap(dedup_list[:int(k)])})
            if ii%10==0: pd.DataFrame(rows).to_csv(RK,index=False)
        rk=pd.DataFrame(rows); rk.to_csv(RK,index=False)
    if os.path.exists(CIp): rkci=pd.read_csv(CIp)
    else:
        rows=[]
        for k in tqdm(sorted(set(list(range(20,len(dedup_list)+1,40))+[len(dedup_list)])),desc=f'CI ({DET})',unit='k'):
            oof=oof_xgb(dedup_list[:k]); m=~np.isnan(oof); md,lo,hi=ap_boot(y[m],oof[m],n=200); rows.append({'k':k,'AP':md,'lo':lo,'hi':hi})
        rkci=pd.DataFrame(rows); rkci.to_csv(CIp,index=False)
    plt.figure(figsize=(11,5)); plt.fill_between(rkci.k,rkci.lo,rkci.hi,color='#2563eb',alpha=.15,label='95% bootstrap CI')
    plt.plot(rk.k,rk.AP_oof,'-',color='#2563eb',lw=1,label='AP OOF folds 1-8 (stable judge)')
    plt.axhline(0.198,ls=':',color='#16a34a',label='M1 OOF 0.198'); plt.axhline(0.299,ls='--',color='#7c3aed',label='M2 OOF 0.299'); plt.axhline(0.619,ls='-.',color='#ea580c',label='M3 OOF 0.619')
    plt.xlabel('k'); plt.ylabel('AP OOF'); plt.legend(); plt.grid(alpha=.3); plt.title(f'M4 {TAG} ({DET}) - AP vs k [OOF scale, all detectors comparable]')
    plt.savefig(os.path.join(FIG,f'm4_{TAG}_ap_vs_k_{DET}.png'),dpi=150,bbox_inches='tight'); plt.show()
    kbest=rk.loc[rk.AP_oof.idxmax()]; K_auto=int(rk[rk.AP_oof>=0.95*kbest.AP_oof].k.min())
    print(f"  max OOF AP {kbest.AP_oof:.3f}@{int(kbest.k)} | K_auto {K_auto} | k_max {len(dedup_list)}")
    G2=os.path.join(PROCESSED,f'm4_{TAG}_grid2d_{DET}.csv'); kmax=len(dedup_list)
    KS=sorted(set(list(range(40,kmax,30))+[K_auto,kmax]))
    CFGS={}
    for dep,lrs in [(2,(0.03,0.05)),(3,(0.03,0.05,0.1)),(4,(0.03,0.05,0.1))]:
        for lr in lrs: CFGS[f'd{dep}_lr{int(lr*100):02d}']=dict(max_depth=dep,learning_rate=lr,n_estimators=200,min_child_weight=3)
    print(f"  KS grid ({len(KS)} pts): {KS}")
    if os.path.exists(G2): G2d=pd.read_csv(G2)
    else:
        rows=[]
        for k in tqdm(KS,desc=f'grid ({DET})',unit='K'):
            for cn,c in CFGS.items():
                ao,at,gp,af=diag_one(c,dedup_list[:k]); rows.append({'K':k,'cfg':cn,'depth':c['max_depth'],'AP_oof':ao,'AP_train':at,'gap':gp,'AP_f9':af})
        G2d=pd.DataFrame(rows); G2d.to_csv(G2,index=False)
    print(f"  AP_train median {G2d.AP_train.median():.3f}")
    mo=float(G2d.AP_oof.max()); elig=G2d[G2d.AP_oof>=mo-TIE_EPS].sort_values(['K','AP_f9'],ascending=[True,False]); best=elig.iloc[0]
    K=int(best.K); CFGNAME=best['cfg']; FEATURES_K=dedup_list[:K]; CFG={**CFGS[CFGNAME],'subsample':0.8,'colsample_bytree':0.8,'reg_lambda':2.0}
    X8,X9=Xall8[:,:K],Xall9[:,:K]
    def make_final(**kw): return make_xgb(spw8,**{**CFG,**kw})
    MODEL_P=os.path.join(MODELS,f'm4_{TAG}_{DET}_model_raw.joblib'); PLATT_P=os.path.join(MODELS,f'm4_{TAG}_{DET}_platt.joblib')
    OOF_P=os.path.join(PROCESSED,f'm4_{TAG}_{DET}_oof.csv'); CFGJ_P=os.path.join(MODELS,f'm4_{TAG}_{DET}_config.json')
    FROZEN=all(os.path.exists(p) for p in (MODEL_P,PLATT_P,OOF_P,CFGJ_P))   # reload the frozen model, no refit -> bit-exact (avoids run-to-run XGBoost drift; refit only if absent)
    if FROZEN:
        print("  [guard] frozen model+oof exist -> reload (no refit; bit-exact).")
        xgb_raw=joblib.load(MODEL_P); platt=joblib.load(PLATT_P); _cj=json.load(open(CFGJ_P,encoding='utf-8'))
        _oo=pd.read_csv(OOF_P,dtype={'ecg_id':str}); _mp=dict(zip(_oo.ecg_id,_oo.proba_raw)); oof_raw=np.array([_mp.get(e,np.nan) for e in eid8],dtype=float)
        mask=~np.isnan(oof_raw); MULTI=_cj.get('multiseed',{}); ap_train=float(_cj.get('metrics_standard_fold9',{}).get('ap_train',float('nan')))
        pf=np.array([average_precision_score(y[(folds==h)&mask],oof_raw[(folds==h)&mask]) for h in sorted(np.unique(folds)) if y[(folds==h)&mask].sum()>0])
    else:
        pf=oof_perfold_ap(FEATURES_K,**CFGS[CFGNAME])
        xgb_raw=make_final().fit(X8,y); ap_train=average_precision_score(y,xgb_raw.predict_proba(X8)[:,1])
        oof_raw=np.full(len(d8),np.nan)
        for trm,vam in FOLD_MASKS:
            if y[trm].sum()==0 or y[vam].sum()==0: continue
            oof_raw[vam]=make_xgb((y[trm]==0).sum()/max((y[trm]==1).sum(),1),**CFG).fit(X8[trm],y[trm]).predict_proba(X8[vam])[:,1]
        mask=~np.isnan(oof_raw); platt=LogisticRegression(max_iter=2000).fit(oof_raw[mask].reshape(-1,1),y[mask])
        aps=[];aucs=[]
        for s in range(10):
            mm=make_final(random_state=s).fit(X8,y); pp=mm.predict_proba(X9)[:,1]; aps.append(average_precision_score(y9,pp)); aucs.append(roc_auc_score(y9,pp))
        MULTI=dict(AP_mean=float(np.mean(aps)),AP_std=float(np.std(aps)),AUC_mean=float(np.mean(aucs)),AUC_std=float(np.std(aucs)))
    print(f"  >>> SELECTED: K={K} {CFGNAME} depth{int(best.depth)} | OOF {best.AP_oof:.3f} fold9 {best.AP_f9:.3f} | per-fold {pf.mean():.3f}+/-{pf.std():.3f} (min {pf.min():.3f})")
    score9=xgb_raw.predict_proba(f9[FEATURES_K] if FROZEN else X9)[:,1]; score9_cal=platt.predict_proba(score9.reshape(-1,1))[:,1]   # pandas frame: matches the frozen model's feature names
    ap_oof_final=average_precision_score(y[mask],oof_raw[mask])
    order=np.argsort(-score9); yy=y9[order]; tp=np.cumsum(yy); rec=tp/max(yy.sum(),1); prec=tp/np.arange(1,len(yy)+1)
    okr=np.where(rec>=0.8)[0]; p80=float(prec[okr[0]]) if len(okr) else float('nan')
    M4_STD=evaluate_standard(name=f'M4_{TAG}_{DET}',y_oof=y[mask],score_oof=oof_raw[mask],y_test=y9,score_test=score9,
        figures_dir=FIG,metrics_dir=METRICS,score_test_calibrated=score9_cal,ap_train=ap_train,multiseed=MULTI,
        extra={'K':K,'cfg':CFGNAME,'depth':int(best.depth),'R_DETECTOR':DET,'OOF_AP_folds1_8':float(ap_oof_final),'precision_at_recall0.8_fold9':p80,'failrate':failrate})
    print(f"  OOF AP {ap_oof_final:.3f} | fold9 AP {M4_STD['AP']:.3f} | AUC {M4_STD['AUC']:.3f} | prec@rec0.8 {p80:.3f}")
    for s in ('ptbxl','ningbo'):
        ms=(src8==s)&mask
        if y[ms].sum()>0: print(f"    OOF AP {s}: {average_precision_score(y[ms],oof_raw[ms]):.3f}")
    print(f"    dataset-confound AUC(source~score) = {roc_auc_score((src8=='ptbxl').astype(int)[mask],oof_raw[mask]):.3f} (0.5=clean)")
    if DEMO:
        ag=np.array([DEMO.get(e,(np.nan,np.nan))[0] for e in eid8]); sx=np.array([DEMO.get(e,(np.nan,np.nan))[1] for e in eid8])
        ma=mask&~np.isnan(ag)
        if ma.sum()>50: print(f"    age-confound : corr(score,age) = {np.corrcoef(oof_raw[ma],ag[ma])[0,1]:+.3f} (PTB, n={int(ma.sum())})")
        msx=mask&~np.isnan(sx)
        if msx.sum()>50 and len(np.unique(sx[msx]))>1: print(f"    sex-confound : AUC(sex~score) = {roc_auc_score(sx[msx],oof_raw[msx]):.3f} (0.5=clean, PTB)")
    else: print("    age/sex confound: skipped (no demographics)")
    mm=d8[['ecg_id','label']].copy(); mm['m4']=oof_raw; have={}
    for k,p in [('m1',os.path.join(PROCESSED,'m1_combined_oof.csv')),('m3',os.path.join(PROCESSED,'m3_combined_oof.csv'))]:
        if os.path.exists(p): mm=mm.merge(pd.read_csv(p,dtype={'ecg_id':str})[['ecg_id','proba_raw']].rename(columns={'proba_raw':k}),on='ecg_id',how='left'); have[k]=1
    p2=os.path.join(ROOT,'models','M2_statistical','m2_oof_folds1_8.npy')
    if os.path.exists(p2):
        a2=np.load(p2)
        if len(a2)==len(mm): mm['m2']=a2; have['m2']=1
    for k in ('m1','m2','m3'):
        if k in have:
            ssub=mm[mm['m4'].notna() & mm[k].notna()]; rho=ssub[['m4',k]].corr(method='spearman').iloc[0,1]
            X=ssub[[k,'m4']].fillna(ssub[[k,'m4']].mean()); apb=average_precision_score(ssub.label,ssub[k])
            st=LogisticRegression(max_iter=1000).fit(X,ssub.label); aps_=average_precision_score(ssub.label,st.predict_proba(X)[:,1])
            print(f"    M4 vs {k.upper()}: rho={rho:.3f} | {k.upper()} {apb:.3f} -> {k.upper()}+M4 {aps_:.3f}")
    if not FROZEN:
        oof_df=d8[['ecg_id','patient_id','fold','source','label']].copy(); oof_df['proba_raw']=oof_raw; oof_df['proba_cal']=platt.predict_proba(oof_raw.reshape(-1,1))[:,1]
        oof_df.to_csv(os.path.join(PROCESSED,f'm4_{TAG}_{DET}_oof.csv'),index=False)
        joblib.dump(xgb_raw,os.path.join(MODELS,f'm4_{TAG}_{DET}_model_raw.joblib')); joblib.dump(platt,os.path.join(MODELS,f'm4_{TAG}_{DET}_platt.joblib'))
        pd.Series(FEATURES_K,name='feature').to_csv(os.path.join(MODELS,f'm4_{TAG}_{DET}_features.csv'),index=False)
        cfg={'model':f'M4_medianbeat_{TAG}','version':'1.0','R_DETECTOR':DET,'K':int(K),'cfg':CFGNAME,'depth':int(best.depth),
             'OOF_AP':float(ap_oof_final),'failrate':failrate,'metrics_standard_fold9':M4_STD,'multiseed':MULTI,
             'representation':'median+most-pre-excited+variability; calibration-invariant morphology; delineation-free (R-peak only)',
             'selection':'gate |d|>0.3 & p_FDR & IC95 & cross-dataset; dedup>0.95; max OOF AP depth-open + parsimony tiebreak; fold10 untouched',
             'fold10':'UNTOUCHED','date_frozen':datetime.now().strftime('%Y-%m-%d %H:%M')}
        json.dump(cfg,open(os.path.join(MODELS,f'm4_{TAG}_{DET}_config.json'),'w',encoding='utf-8'),ensure_ascii=False,indent=2)
        print(f"  M4 ({DET}) FROZEN.")
    else:
        print(f"  M4 ({DET}) reloaded from frozen v1 (artifacts untouched).")
    return {'DET':DET,'OOF_AP':float(ap_oof_final),'fold9_AP':float(M4_STD['AP']),'AUC':float(M4_STD['AUC']),'K':K,'failrate':failrate}
print("Block 8a.2: run_detector() defined.")

### Block 8a.3: Driver — Run All runs BOTH R-detectors (A/B), resumable

In [ ]:
# Driver: ONE Run All runs BOTH R-detectors end-to-end (A/B), resumable. When the caches are present the guards
# reload them in minutes (reproducing pan_tompkins 0.705 / wavelet_env 0.718) without re-extracting.
SUMMARIES={}
for DET in DETECTORS:
    SUMMARIES[DET]=run_detector(DET)
print("\n=== both detectors done ===")
for DET,s in SUMMARIES.items(): print(f"  {DET}: OOF AP {s['OOF_AP']:.3f} | fold9 {s['fold9_AP']:.3f} | AUC {s['AUC']:.3f} | K {s['K']} | failrate {100*s['failrate']:.1f}%")

### Block 8a.4: Permutation negative-control (shuffle labels → null OOF AP must collapse)

In [ ]:
# Permutation negative-control (method check): shuffle train labels (folds 1-8, prevalence preserved) -> top-K
# features by shuffled |d| should give an OOF AP collapsing to ~chance. Proves the gate+selection does not
# manufacture signal from noise. Cached. Run on the A/B-winning detector.
PERM_DET='wavelet_env' if 'wavelet_env' in DETECTORS else DETECTORS[0]
PERM_CSV=os.path.join(PROCESSED,f'm4_{TAG}_permutation_{PERM_DET}.csv')
_p=os.path.join(PROCESSED,f'm4_features_{PERM_DET}.csv'); _h=pd.read_csv(_p,nrows=0).columns.tolist()
_sc={'ecg_id','patient_id','source'}; _ic={'label','fold','extraction_failed','n_beats'}
dfp=pd.read_csv(_p,dtype={c:(str if c in _sc else 'int16' if c in _ic else 'float32') for c in _h})
METAC=['ecg_id','patient_id','label','fold','source','extraction_failed','n_beats']
FEATp=[c for c in dfp.columns if c not in METAC]
d8p=dfp[dfp.fold.between(1,8)].reset_index(drop=True); yp=d8p.label.values.copy(); foldsp=d8p.fold.values
FMp=[(foldsp!=h,foldsp==h) for h in sorted(np.unique(foldsp))]
def _oof_ap_perm(feats,yv):
    X=d8p[feats]; oof=np.full(len(d8p),np.nan)
    for trm,vam in FMp:
        if yv[trm].sum()==0 or yv[vam].sum()==0: continue
        oof[vam]=make_xgb((yv[trm]==0).sum()/max((yv[trm]==1).sum(),1)).fit(X[trm],yv[trm]).predict_proba(X[vam])[:,1]
    m=~np.isnan(oof); return float(average_precision_score(yv[m],oof[m]))
if os.path.exists(PERM_CSV):
    pr=pd.read_csv(PERM_CSV); print("[guard] permutation reloaded.")
else:
    rng=np.random.default_rng(123); rows=[]
    for it in tqdm(range(5),desc='permutation',unit='shuffle'):
        ysh=yp.copy(); rng.shuffle(ysh)
        ds=np.array([abs(cohens_d(d8p[c].values[ysh==1].astype(float),d8p[c].values[ysh==0].astype(float))) for c in FEATp])
        top=[FEATp[i] for i in np.argsort(-np.nan_to_num(ds))[:50]]
        rows.append({'shuffle':it,'null_OOF_AP':_oof_ap_perm(top,ysh)})
    pr=pd.DataFrame(rows); pr.to_csv(PERM_CSV,index=False)
real=SUMMARIES[PERM_DET]['OOF_AP']; prev=float(yp.mean())
print(f"=== Permutation negative-control ({PERM_DET}) ===")
print(f"  null OOF AP (5 shuffles): mean {pr.null_OOF_AP.mean():.3f}, max {pr.null_OOF_AP.max():.3f} | prevalence {prev:.3f} | REAL OOF AP {real:.3f}")
print(f"  -> {'PASS: real >> null (no manufactured signal)' if real>pr.null_OOF_AP.max()+0.1 else 'CHECK: real not clearly above null'}")

### Block 8a.5: R-detector concordance (pan_tompkins vs wavelet_env agree on R?)

In [ ]:
# R-detector concordance: on a sample, do pan_tompkins and wavelet_env agree on R locations? (reliability check)
samp=meta.sample(min(400,len(meta)),random_state=42).to_dict('records'); frac=[]; dcnt=[]
for m in tqdm(samp,desc='R concordance',unit='ecg'):
    try:
        raw=load_signal(m['ecg_id'],m['source']); bp12=np.column_stack([bp(raw[:,LEAD_IDX[L]]) for L in LEADS_M4])
        rp=detect_r(bp12,'pan_tompkins'); rw=detect_r(bp12,'wavelet_env')
        if len(rp)==0 or len(rw)==0: continue
        tol=int(0.025*FS); ok=sum(1 for r in rp if np.min(np.abs(rw-r))<=tol)
        frac.append(ok/len(rp)); dcnt.append(abs(len(rp)-len(rw)))
    except Exception: continue
print(f"=== R-detector concordance (n={len(frac)}) ===")
print(f"  PT peaks within 25ms of a wavelet_env peak: {np.mean(frac):.1%} | mean |#R_pt - #R_we|: {np.mean(dcnt):.2f}")
print("  -> high agreement = detection reliable; the A/B is about feature/perf, not detection disagreement.")

### Block 8a.6: Verdict — pick R-detector + promote winner to canonical m4_combined_*

In [ ]:
# A/B VERDICT: pick by OOF AP, then lower rho(M4,M3), then lower failrate. Promote winner to canonical
# m4_combined_* (deployed M4; used by the ensemble notebook). Fold 10 untouched.
import shutil
m3=None
if os.path.exists(os.path.join(PROCESSED,'m3_combined_oof.csv')):
    m3=pd.read_csv(os.path.join(PROCESSED,'m3_combined_oof.csv'),dtype={'ecg_id':str})[['ecg_id','proba_raw']].rename(columns={'proba_raw':'m3'})
rows=[]
for DET in DETECTORS:
    s=SUMMARIES[DET]; oo=pd.read_csv(os.path.join(PROCESSED,f'm4_{TAG}_{DET}_oof.csv'),dtype={'ecg_id':str}); rho=np.nan
    if m3 is not None:
        j=oo.merge(m3,on='ecg_id',how='inner'); rho=float(j[['proba_raw','m3']].corr(method='spearman').iloc[0,1])
    rows.append({'DET':DET,'OOF_AP':s['OOF_AP'],'fold9_AP':s['fold9_AP'],'AUC':s['AUC'],'failrate':s['failrate'],'rho_M3':rho})
ab=pd.DataFrame(rows); pd.set_option('display.float_format',lambda x:f'{x:.3f}')
print("=== A/B (R-detector) ==="); print(ab.to_string(index=False))
ab=ab.sort_values(['OOF_AP','rho_M3','failrate'],ascending=[False,True,True]); win=ab.iloc[0]['DET']
print(f"\n>>> WINNER: {win}")
for ext in ['model_raw.joblib','platt.joblib','features.csv','config.json']:
    s=os.path.join(MODELS,f'm4_{TAG}_{win}_{ext}'); d=os.path.join(MODELS,f'm4_{TAG}_{ext}')
    if os.path.exists(s): shutil.copy(s,d)
shutil.copy(os.path.join(PROCESSED,f'm4_{TAG}_{win}_oof.csv'),os.path.join(PROCESSED,f'm4_{TAG}_oof.csv'))
cfgp=os.path.join(MODELS,f'm4_{TAG}_config.json'); cfg=json.load(open(cfgp,encoding='utf-8')); cfg['winner_detector']=win; cfg['ab_table']=ab.to_dict('records')
json.dump(cfg,open(cfgp,'w',encoding='utf-8'),ensure_ascii=False,indent=2)
print(f"  promoted {win} -> canonical m4_combined_* (model, platt, features, config, oof). Fold 10 untouched.")

### Results & interpretation

v1 frozen (reload-mode → bit-exact): pan_tompkins OOF 0.705 (K=273, fold9 TP10/FN3) / **wavelet_env OOF 0.718 (K=220, depth3, fold9 TP10/FN3)** → winner wavelet_env. Per-fold 0.732±0.054 (min 0.609); fold9 AP 0.837, AUC 0.960; dataset-confound 0.522; M4↔M3 rho 0.237 → M3+M4 0.736; permutation null ≈0. Read: family ablation (delta_onset / shape carry M4; variability is weak), orthogonality to M3 (persistent vs intermittent), confounds. Final = fold 10 (the held-out test).